In [1]:
import sys; sys.path.append("../src")
import torch
from data_prep import prepare_data
from models import IDSNet
from attacks import fgsm, pgd

d = prepare_data("../data/KDDTrain.txt", "../data/KDDTest.txt")
X_val = torch.tensor(d["X_val"]); y_val = torch.tensor(d["y_val"], dtype=torch.long)

baseline = IDSNet(X_val.shape[1]); baseline.load_state_dict(torch.load("../saved_models/nn_baseline.pt"))
hardened = IDSNet(X_val.shape[1]); hardened.load_state_dict(torch.load("../saved_models/nn_hardened.pt"))
baseline.eval(); hardened.eval()

# Features an attacker can realistically change: padding bytes, slowing the connection,
# and changing how many connections they open. Protocol/service/flag and exploit-content
# features are fixed, because changing them would break the attack itself.
CONTROLLABLE = ["duration", "src_bytes", "count", "srv_count", "dst_host_count", "dst_host_srv_count"]
mask = torch.tensor([1.0 if n in CONTROLLABLE else 0.0 for n in d["feature_names"]])
print("controllable features:", int(mask.sum()), "of", len(mask))

controllable features: 6 of 121


In [2]:
def evasion(model, attack, eps):
    with torch.no_grad():
        caught = model(X_val).argmax(dim=1) == 1
    m = (y_val == 1) & caught
    Xa = attack(model, X_val[m], y_val[m], eps)
    with torch.no_grad():
        return (model(Xa).argmax(dim=1) == 0).float().mean().item() * 100

attacks = {
    "FGSM": lambda m, X, y, e: fgsm(m, X, y, e),
    "PGD": lambda m, X, y, e: pgd(m, X, y, e),
    "PGD-feasible": lambda m, X, y, e: pgd(m, X, y, e, mask=mask),
}

print(f"{'eps':>5} | " + " | ".join(f"base {n:>12}" for n in attacks) + " | " + " | ".join(f"hard {n:>12}" for n in attacks))
for eps in [0.05, 0.1, 0.2, 0.5, 1.0]:
    row = [evasion(baseline, a, eps) for a in attacks.values()] + [evasion(hardened, a, eps) for a in attacks.values()]
    print(f"{eps:>5.2f} | " + " | ".join(f"{v:>17.1f}%" for v in row))

  eps | base         FGSM | base          PGD | base PGD-feasible | hard         FGSM | hard          PGD | hard PGD-feasible
 0.05 |              27.8% |              29.9% |               0.1% |               0.1% |               0.2% |               0.0%
 0.10 |              97.1% |              97.5% |               0.2% |               0.5% |               0.7% |               0.0%
 0.20 |              99.6% |              99.9% |               0.8% |               3.4% |               6.3% |               0.2%
 0.50 |             100.0% |             100.0% |               4.9% |              92.9% |              96.6% |               1.0%
 1.00 |              99.9% |             100.0% |              13.2% |              97.9% |             100.0% |               4.5%
